# 4 · CNN de Ingredientes — Reentrenamiento con Dataset Extendido

## Problema
El modelo actual solo reconoce **183 clases** entrenadas con **Fruits-360** (fotos de catálogo sobre fondo blanco).  
Las fotos reales del usuario fallan porque el modelo nunca vio ese estilo de imagen.

## Solución
Combinar **3 datasets** con estilos visuales diferentes:

| Dataset | Clases | Imágenes | Fondo | Uso |
|---|---|---|---|---|
| Fruits-360 (`moltean/fruits`) | ~260 | ~90k | Blanco artificial | Frutas y verduras básicas |
| Vegetable Images (`misrakahmed/vegetable-image-dataset`) | 15 | ~21k | **Fondo real** | Verduras comunes de cocina |
| Fruit & Veg Recognition (`kritikseth/fruit-and-vegetable-image-recognition`) | 36 | ~3k | **Fondo real** | Amplia variedad |
| Recipe Ingredients (`fasihcs/recipe-ingredients-image-dataset`) | ~42 | variable | Mixto | Ingredientes de despensa |

**Resultado esperado:** ~250–300 clases con imágenes reales → mejor generalización en fotos de usuarios.

---
> ⚡ Ejecutar en **Google Colab con GPU T4 o mejor** (Runtime → Change runtime type → GPU)

In [ ]:
# ── Verificar GPU ─────────────────────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU disponible:', result.stdout.strip())
else:
    print('⚠️  No se detectó GPU. Ve a Runtime → Change runtime type → GPU')

In [ ]:
# ── Instalar dependencias ──────────────────────────────────────────────────────
!pip install kagglehub huggingface_hub timm -q
print('Dependencias listas ✅')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json, os, re, shutil, time
from collections import defaultdict
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import HfApi, login

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
WORK_DIR   = Path('/content/cnn_extended')
WORK_DIR.mkdir(exist_ok=True)
print(f'Directorio de trabajo: {WORK_DIR}')

---
## 1 · Descarga de Datasets

In [ ]:
# ── Dataset 1: Fruits-360 ─────────────────────────────────────────────────────
print('Descargando Fruits-360...')
path_fruits = Path(kagglehub.dataset_download('moltean/fruits'))
# Localizar carpeta Training
fruits_train = next(path_fruits.rglob('Training'), None)
print(f'  Fruits-360 Training: {fruits_train}')
print(f'  Clases: {len(list(fruits_train.iterdir()))}' if fruits_train else '  No encontrado')

In [ ]:
# ── Dataset 2: Vegetable Image Dataset (fotos reales, 15 clases) ──────────────
print('Descargando Vegetable Image Dataset...')
path_veg = Path(kagglehub.dataset_download('misrakahmed/vegetable-image-dataset'))
print(f'  Path: {path_veg}')

# Explorar estructura
for split_dir in sorted(path_veg.rglob('train')):
    classes_veg = [d.name for d in split_dir.iterdir() if d.is_dir()]
    n_imgs = sum(len(list(d.iterdir())) for d in split_dir.iterdir() if d.is_dir())
    print(f'  Split train: {len(classes_veg)} clases | {n_imgs:,} imágenes')
    print(f'  Clases: {classes_veg}')
    veg_train = split_dir
    break

In [ ]:
# ── Dataset 3: Fruit and Vegetable Recognition (fotos reales, 36 clases) ──────
print('Descargando Fruit and Vegetable Recognition...')
path_fvr = Path(kagglehub.dataset_download('kritikseth/fruit-and-vegetable-image-recognition'))
print(f'  Path: {path_fvr}')

for split_dir in sorted(path_fvr.rglob('train')):
    classes_fvr = [d.name for d in split_dir.iterdir() if d.is_dir()]
    n_imgs = sum(len(list(d.iterdir())) for d in split_dir.iterdir() if d.is_dir())
    print(f'  Split train: {len(classes_fvr)} clases | {n_imgs:,} imágenes')
    print(f'  Clases: {classes_fvr}')
    fvr_train = split_dir
    break

In [ ]:
# ── Dataset 4: Recipe Ingredients Images (ingredientes de despensa) ────────────
print('Descargando Recipe Ingredients Images...')
path_ingr = Path(kagglehub.dataset_download('fasihcs/recipe-ingredients-image-dataset'))
print(f'  Path: {path_ingr}')

# Recolectar carpetas con imágenes
ingr_dirs = {}
for d in sorted(path_ingr.rglob('*')):
    if d.is_dir():
        n = sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
        if n > 0:
            ingr_dirs[d] = n

print(f'  Carpetas con imágenes: {len(ingr_dirs)}')
print(f'  Total imágenes: {sum(ingr_dirs.values()):,}')
for d, n in sorted(ingr_dirs.items(), key=lambda x: -x[1])[:10]:
    print(f'    {d.name:<35} {n} imgs')

---
## 2 · Construcción del Dataset Unificado

Estrategia de normalización de nombres de clase:
- Minúsculas, sin números, sin guiones → `"Apple Braeburn 1"` → `"apple braeburn"`
- Clases de múltiples fuentes se **fusionan** bajo el mismo nombre normalizado
- Máximo **800 imágenes** por clase (para evitar desequilibrio extremo)
- Mínimo **30 imágenes** por clase (se descartan clases con muy pocas muestras)

In [ ]:
# ── Función de normalización ──────────────────────────────────────────────────
def normalize_class(name: str) -> str:
    """Convierte nombre de carpeta a clave de clase normalizada."""
    name = name.lower().strip()
    name = re.sub(r'\d+', '', name)       # quitar números
    name = re.sub(r'[-_]', ' ', name)     # guiones → espacios
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Prueba
for test in ['Apple Braeburn 1', 'Tomato 3', 'bitter_gourd', 'Potato', 'chicken-breast']:
    print(f'  {test!r:<30} → {normalize_class(test)!r}')

In [ ]:
# ── Construir catálogo de todas las fuentes ───────────────────────────────────
# class_sources: { normalized_name: [lista de rutas de imagen] }
class_sources: dict[str, list[Path]] = defaultdict(list)

def collect_images_from_dir(class_dir: Path) -> list[Path]:
    return sorted(f for f in class_dir.iterdir()
                  if f.is_file() and f.suffix.lower() in IMAGE_EXTS)

# ── Fuente 1: Fruits-360 ──────────────────────────────────────────────────────
n_fruits = 0
for cls_dir in sorted(fruits_train.iterdir()):
    if not cls_dir.is_dir(): continue
    key  = normalize_class(cls_dir.name)
    imgs = collect_images_from_dir(cls_dir)
    if imgs:
        class_sources[key].extend(imgs)
        n_fruits += 1
print(f'Fruits-360:  {n_fruits} clases añadidas')

# ── Fuente 2: Vegetable Image Dataset ─────────────────────────────────────────
n_veg = 0
for cls_dir in sorted(veg_train.iterdir()):
    if not cls_dir.is_dir(): continue
    key  = normalize_class(cls_dir.name)
    imgs = collect_images_from_dir(cls_dir)
    if imgs:
        class_sources[key].extend(imgs)
        n_veg += 1
print(f'Vegetable:   {n_veg} clases (algunas fusionadas con Fruits-360)')

# ── Fuente 3: Fruit & Veg Recognition ─────────────────────────────────────────
n_fvr = 0
for cls_dir in sorted(fvr_train.iterdir()):
    if not cls_dir.is_dir(): continue
    key  = normalize_class(cls_dir.name)
    imgs = collect_images_from_dir(cls_dir)
    if imgs:
        class_sources[key].extend(imgs)
        n_fvr += 1
print(f'FVR:         {n_fvr} clases (algunas fusionadas)')

# ── Fuente 4: Recipe Ingredients ──────────────────────────────────────────────
n_ingr = 0
for cls_dir, _ in ingr_dirs.items():
    key  = normalize_class(cls_dir.name)
    imgs = collect_images_from_dir(cls_dir)
    if imgs:
        class_sources[key].extend(imgs)
        n_ingr += 1
print(f'Ingredientes:{n_ingr} clases')

print(f'\nTotal clases únicas: {len(class_sources)}')
print(f'Total imágenes brutas: {sum(len(v) for v in class_sources.values()):,}')

In [ ]:
# ── Filtrar y balancear clases ────────────────────────────────────────────────
MIN_IMAGES   = 30    # descartar clases con menos de N imágenes
MAX_IMAGES   = 800   # cap para evitar desequilibrio extremo

filtered: dict[str, list[Path]] = {}
for cls, imgs in class_sources.items():
    # Deduplicar (por si una imagen aparece en múltiples fuentes)
    imgs_unique = list(dict.fromkeys(imgs))
    if len(imgs_unique) < MIN_IMAGES:
        continue
    # Cap aleatorio para balancear
    if len(imgs_unique) > MAX_IMAGES:
        rng = np.random.default_rng(42)
        imgs_unique = list(rng.choice(imgs_unique, MAX_IMAGES, replace=False))
    filtered[cls] = imgs_unique

# Ordenar alfabéticamente para reproducibilidad
class_list = sorted(filtered.keys())
class_to_idx = {c: i for i, c in enumerate(class_list)}

print(f'Clases antes del filtro : {len(class_sources)}')
print(f'Clases después del filtro: {len(filtered)}')
print(f'Total imágenes (balanceado): {sum(len(v) for v in filtered.values()):,}')

# Distribución de tamaños
sizes = sorted([len(v) for v in filtered.values()])
print(f'Imágenes por clase — min: {min(sizes)} | mediana: {int(np.median(sizes))} | max: {max(sizes)}')

# Muestra de clases
print(f'\nPrimeras 30 clases:')
for cls in class_list[:30]:
    print(f'  {cls:<35} {len(filtered[cls])} imgs')

In [ ]:
# ── Generar class_labels.json e ingredient_catalog.json ───────────────────────
class_labels = {str(i): name for i, name in enumerate(class_list)}

# Para el catálogo, tomar la primera imagen de cada clase como representativa
ingredient_catalog = {}
for cls in class_list:
    imgs = filtered[cls]
    # Preferir imágenes del dataset de vegetales (fondo real) si existen
    veg_imgs = [p for p in imgs if 'vegetable' in str(p).lower() or 'kritikseth' in str(p).lower()]
    representative = veg_imgs[0] if veg_imgs else imgs[0]
    ingredient_catalog[cls] = str(representative)

print(f'class_labels: {len(class_labels)} clases')
print(f'ingredient_catalog: {len(ingredient_catalog)} entradas')

# Guardar
with open(WORK_DIR / 'class_labels.json', 'w') as f:
    json.dump(class_labels, f, indent=2, ensure_ascii=False)
with open(WORK_DIR / 'ingredient_catalog.json', 'w') as f:
    json.dump(ingredient_catalog, f, indent=2, ensure_ascii=False)

print('Guardados en', WORK_DIR)

---
## 3 · Dataset y Transformaciones

Usar **augmentación agresiva** para que el modelo aprenda a generalizar a:
- Fondos variados (no solo blanco)
- Diferentes iluminaciones
- Distintos ángulos y recortes

In [ ]:
# ── Transformaciones ──────────────────────────────────────────────────────────
from torchvision.transforms import v2

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),   # más variación de escala
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.1),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print('Transformaciones definidas ✅')

In [ ]:
# ── Clase Dataset ─────────────────────────────────────────────────────────────
class IngredientDataset(Dataset):
    """Dataset combinado de múltiples fuentes de imágenes de ingredientes."""

    def __init__(self, samples: list[tuple[Path, int]], transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), (128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img, label


# ── Construir lista de samples y split 85/15 ──────────────────────────────────
all_samples: list[tuple[Path, int]] = []
for cls in class_list:
    label = class_to_idx[cls]
    for img_path in filtered[cls]:
        all_samples.append((img_path, label))

# Mezclar con seed fija
rng = np.random.default_rng(42)
idx_perm = rng.permutation(len(all_samples)).tolist()
all_samples = [all_samples[i] for i in idx_perm]

n_val    = int(len(all_samples) * 0.15)
n_train  = len(all_samples) - n_val
train_samples = all_samples[:n_train]
val_samples   = all_samples[n_train:]

train_ds = IngredientDataset(train_samples, transform=train_tf)
val_ds   = IngredientDataset(val_samples,   transform=val_tf)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Clases: {len(class_list)}')

In [ ]:
# ── DataLoaders con WeightedRandomSampler para balancear clases ───────────────
BATCH_SIZE = 128
NUM_WORKERS = 4

# Pesos inversamente proporcionales al tamaño de cada clase
class_counts = defaultdict(int)
for _, label in train_samples:
    class_counts[label] += 1

sample_weights = [1.0 / class_counts[label] for _, label in train_samples]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_samples), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Batch size: {BATCH_SIZE}')

---
## 4 · Modelo y Entrenamiento

In [ ]:
# ── Construir modelo ──────────────────────────────────────────────────────────
NUM_CLASSES = len(class_list)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(in_features, NUM_CLASSES),
)
model = model.to(DEVICE)

print(f'EfficientNet-B0 → {NUM_CLASSES} clases')
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros: {total_params:,} total | {trainable:,} entrenables')

In [ ]:
# ── Configuración de entrenamiento ────────────────────────────────────────────
EPOCHS    = 20
LR_INIT   = 1e-3
LR_WARMUP = 5    # épocas de warmup lineal antes de cosine decay

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_INIT, weight_decay=1e-4)

# Warmup lineal + CosineAnnealing
def lr_lambda(epoch):
    if epoch < LR_WARMUP:
        return (epoch + 1) / LR_WARMUP
    progress = (epoch - LR_WARMUP) / max(EPOCHS - LR_WARMUP, 1)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f'Épocas: {EPOCHS} | LR inicial: {LR_INIT} | Warmup: {LR_WARMUP} épocas')

In [ ]:
# ── Loop de entrenamiento ─────────────────────────────────────────────────────
WEIGHTS_PATH = WORK_DIR / 'efficientnet_ingredients.pth'

best_val_acc = 0.0
history = []

print(f"{'Época':>5} {'Loss Tr':>9} {'Acc Tr':>8} {'Loss Val':>9} {'Acc Val':>8} {'LR':>9}")
print('─' * 60)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        tr_loss    += loss.item() * len(imgs)
        tr_correct += (logits.argmax(1) == labels).sum().item()
        tr_total   += len(imgs)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    va_loss, va_correct, va_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            va_loss    += loss.item() * len(imgs)
            va_correct += (logits.argmax(1) == labels).sum().item()
            va_total   += len(imgs)

    scheduler.step()
    lr = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0

    tr_acc = tr_correct / tr_total
    va_acc = va_correct / va_total

    row = dict(epoch=epoch, tr_loss=tr_loss/tr_total, tr_acc=tr_acc,
               va_loss=va_loss/va_total, va_acc=va_acc, lr=lr)
    history.append(row)

    print(f"{epoch:>5} {row['tr_loss']:>9.4f} {tr_acc:>7.1%} "
          f"{row['va_loss']:>9.4f} {va_acc:>7.1%} {lr:>9.2e}  ({elapsed:.0f}s)")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), WEIGHTS_PATH)
        print(f"         ↑ guardado  val_acc={va_acc:.2%}")

print(f'\nEntrenamiento completo. Mejor val_acc: {best_val_acc:.2%}')
print(f'Pesos guardados en: {WEIGHTS_PATH}')

---
## 5 · Evaluación Detallada

In [ ]:
# ── Cargar mejor modelo ───────────────────────────────────────────────────────
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

# ── Curva de entrenamiento ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

df_hist = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df_hist['epoch'], df_hist['tr_loss'], label='Train')
axes[0].plot(df_hist['epoch'], df_hist['va_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Época')
axes[0].legend()
axes[1].plot(df_hist['epoch'], df_hist['tr_acc'],  label='Train')
axes[1].plot(df_hist['epoch'], df_hist['va_acc'],  label='Val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Época')
axes[1].legend()
plt.suptitle(f'EfficientNet-B0 · {NUM_CLASSES} clases · mejor val_acc={best_val_acc:.2%}')
plt.tight_layout()
plt.savefig(WORK_DIR / 'training_curves.png', dpi=120)
plt.show()
print(f'Gráfica guardada en {WORK_DIR}/training_curves.png')

In [ ]:
# ── Top-5 accuracy y clases con más errores ───────────────────────────────────
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        top5   = logits.topk(5, dim=1).indices.cpu()
        all_preds.append(top5)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

top1_acc = (all_preds[:, 0] == all_labels).float().mean().item()
top5_acc = (all_labels.unsqueeze(1) == all_preds).any(dim=1).float().mean().item()

print(f'Top-1 accuracy: {top1_acc:.2%}')
print(f'Top-5 accuracy: {top5_acc:.2%}')

# Clases con más errores (útil para identificar qué mejorar)
errors = defaultdict(int)
totals = defaultdict(int)
for pred, label in zip(all_preds[:, 0].tolist(), all_labels.tolist()):
    totals[label] += 1
    if pred != label:
        errors[label] += 1

error_rates = {class_list[k]: errors[k] / totals[k] for k in totals if totals[k] > 0}
top_errors = sorted(error_rates.items(), key=lambda x: -x[1])[:20]

print(f'\nTop 20 clases con más errores:')
for cls, rate in top_errors:
    n = totals[class_to_idx[cls]]
    print(f'  {cls:<35} error: {rate:.0%}  ({n} muestras val)')

---
## 6 · Subir a Hugging Face Hub

In [ ]:
# ── Login ─────────────────────────────────────────────────────────────────────
# Generar token en: https://huggingface.co/settings/tokens (rol: write)
HF_USERNAME = 'ramonsj11'   # ← tu username de HuggingFace
HF_TOKEN    = ''            # ← pegar aquí tu token HF o usar getpass

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass('HuggingFace token: ')

login(token=HF_TOKEN)
api = HfApi()
print('Login OK ✅')

In [ ]:
# ── Crear repo y subir archivos ────────────────────────────────────────────────
REPO_ID = f'{HF_USERNAME}/recipe-ingredient-classifier'

api.create_repo(repo_id=REPO_ID, exist_ok=True, repo_type='model')
print(f'Repo: https://huggingface.co/{REPO_ID}')

# Subir pesos del modelo
api.upload_file(
    path_or_fileobj=str(WEIGHTS_PATH),
    path_in_repo='efficientnet_ingredients.pth',
    repo_id=REPO_ID,
)
print('✅ efficientnet_ingredients.pth subido')

# Subir class_labels.json
api.upload_file(
    path_or_fileobj=str(WORK_DIR / 'class_labels.json'),
    path_in_repo='class_labels.json',
    repo_id=REPO_ID,
)
print('✅ class_labels.json subido')

# Subir ingredient_catalog.json (para el Space)
api.upload_file(
    path_or_fileobj=str(WORK_DIR / 'ingredient_catalog.json'),
    path_in_repo='ingredient_catalog.json',
    repo_id=REPO_ID,
)
print('✅ ingredient_catalog.json subido')

print(f'\n🎉 Subida completa → {REPO_ID}')
print(f'   Clases: {NUM_CLASSES} | Mejor val_acc: {best_val_acc:.2%}')

---
## 7 · Actualizar el Space de Hugging Face

Después de subir el modelo, hay que copiar los archivos actualizados al Space.  
El `app_optimized.py` ya descarga los pesos desde el repo del modelo automáticamente.

Solo necesitas actualizar `class_labels.json` e `ingredient_catalog.json` en el Space si los tienes allí de forma estática. Si el Space los descarga del repo del modelo, no hace falta nada más.

**Pasos:**
1. Descargar `class_labels.json` e `ingredient_catalog.json` desde Colab
2. Reemplazarlos en `space_repo/` de tu repositorio local
3. Ejecutar `deploy_to_hf.py` o hacer push manual

In [ ]:
# ── Descargar archivos a Google Drive (opcional) ──────────────────────────────
try:
    from google.colab import drive, files
    print('Descargando archivos...')
    files.download(str(WEIGHTS_PATH))
    files.download(str(WORK_DIR / 'class_labels.json'))
    files.download(str(WORK_DIR / 'ingredient_catalog.json'))
    files.download(str(WORK_DIR / 'training_curves.png'))
    print('✅ Archivos descargados')
except ImportError:
    print('No estás en Colab — los archivos están en', WORK_DIR)

---
## Resumen

| Paso | Descripción |
|---|---|
| 1 | Descarga de 4 datasets (Fruits-360, Vegetable, FVR, Recipe Ingredients) |
| 2 | Normalización y fusión de clases duplicadas |
| 3 | Filtrado (≥30 imgs/clase) y balanceo (≤800 imgs/clase) |
| 4 | Entrenamiento EfficientNet-B0 con augmentación agresiva (20 épocas) |
| 5 | Evaluación Top-1 / Top-5 accuracy |
| 6 | Subida de pesos y metadatos a Hugging Face Hub |

**Resultado esperado:** Top-1 accuracy ≥ 75% en imágenes de fondo mixto.